# Exercise 2 (Starter) — Domain Feature Engineering & Leak-Free `ColumnTransformer`

**Timebox:** 25 minutes
**Goal:** Build your own feature engineering function and `ColumnTransformer` preprocessor for StreamPulse, and prove it handles missing values and unseen categories without leaking test data.

### Your Tasks
1. **Task 1 (`engineer_churn_features`)**: Add three domain columns to a copy of the DataFrame:
   - `charges_per_login = monthly_charges / (login_days_30d + 1.0)`
   - `high_ticket_flag = (support_tickets_30d >= 3).astype(int)`
   - `engagement_ratio = watch_hours_30d / (monthly_charges + 1.0)`
2. **Task 2 (`build_preprocessor`)**: Build and return a `ColumnTransformer` with:
   - Numeric pipeline: `SimpleImputer(strategy="median")` → `StandardScaler()`
   - Categorical pipeline: `SimpleImputer(strategy="most_frequent")` → `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`
3. **Task 3**: Split `X, y` with `stratify=y` and `random_state=42`, fit `preprocessor` **only on `X_train`**, transform `X_train` and `X_test`, and pass all self-check assertions!


In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load and merge raw StreamPulse data
cust = pd.read_csv("data/customers_raw.csv")
with open("data/usage_events.json", "r", encoding="utf-8") as f:
    usage = pd.DataFrame(json.load(f))
cust["plan_tier"] = cust["plan_tier"].apply(lambda x: x.strip().lower() if isinstance(x, str) else x)
cust["country"] = cust["country"].apply(lambda x: x.strip().upper() if isinstance(x, str) else x)
df = cust.drop_duplicates(subset=["customer_id"], keep="first").merge(usage, on="customer_id", how="left")

print("Loaded df shape:", df.shape)


In [ ]:
# ============================================================================
# TASK 1: Engineer 3 domain features
# ============================================================================
def engineer_churn_features(input_df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of input_df with 3 new domain features."""
    out = input_df.copy()

    # TODO 1a: out["charges_per_login"] = ...
    # TODO 1b: out["high_ticket_flag"] = ... (1 if support_tickets_30d >= 3 else 0)
    # TODO 1c: out["engagement_ratio"] = ... (watch_hours_30d / (monthly_charges + 1.0))

    raise NotImplementedError("Complete Task 1 inside engineer_churn_features()")
    return out


In [ ]:
# ============================================================================
# TASK 2: Build the ColumnTransformer preprocessor
# ============================================================================
NUMERIC_COLS = [
    "age", "tenure_months", "monthly_charges",
    "login_days_30d", "watch_hours_30d", "support_tickets_30d",
    "auto_pay", "charges_per_login", "high_ticket_flag", "engagement_ratio",
]
CATEGORICAL_COLS = ["plan_tier", "country", "payment_method", "last_active_device"]


def build_preprocessor(numeric_cols: list[str], categorical_cols: list[str]) -> ColumnTransformer:
    """Create a ColumnTransformer for numeric + categorical features."""
    # TODO 2a: Build numeric_pipe with SimpleImputer(strategy="median") and StandardScaler()
    # TODO 2b: Build categorical_pipe with SimpleImputer(strategy="most_frequent") and
    #          OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    # TODO 2c: Combine them inside a ColumnTransformer and return it

    raise NotImplementedError("Complete Task 2 inside build_preprocessor()")


In [ ]:
# ============================================================================
# TASK 3 & SELF-CHECK TESTS
# ============================================================================
df_fe = engineer_churn_features(df)
X = df_fe[NUMERIC_COLS + CATEGORICAL_COLS]
y = df_fe["churned"]

# TODO 3: Split into X_train, X_test, y_train, y_test (test_size=0.2, random_state=42, stratify=y)
#         Then build preprocessor, fit_transform on X_train, and transform on X_test!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
preprocessor = build_preprocessor(NUMERIC_COLS, CATEGORICAL_COLS)
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

# Automated self-checks
assert {"charges_per_login", "high_ticket_flag", "engagement_ratio"}.issubset(df_fe.columns)
assert set(df_fe["high_ticket_flag"].unique()).issubset({0, 1}), "high_ticket_flag must be binary 0/1"
assert X_train_t.shape[0] == len(X_train)
assert X_test_t.shape[0] == len(X_test)
assert X_train_t.shape[1] == X_test_t.shape[1], "Train and test must have identical feature counts!"
assert not np.isnan(X_train_t).any(), "NaNs found in X_train_t!"
assert not np.isnan(X_test_t).any(), "NaNs found in X_test_t!"

# Verify unseen category safety
probe = X_test.iloc[[0]].copy()
probe["plan_tier"] = "future_tier_2027"
probe["monthly_charges"] = np.nan
probe_t = preprocessor.transform(probe)
assert not np.isnan(probe_t).any(), "Preprocessor failed on unseen category / missing value!"

print(f"[ALL TESTS PASSED] Transformed matrix shape: train={X_train_t.shape}, test={X_test_t.shape}")


### Bonus Stretch Task
Add a `tenure_bucket` categorical feature using `pd.cut(tenure_months, bins=[0, 6, 18, 100], labels=["new", "growing", "loyal"])`, include it in `CATEGORICAL_COLS`, and inspect `preprocessor.get_feature_names_out()`.
